# Advanced Experiment Tracking: The Full Ecosystem

## Beyond MLflow and W&B

MLflow and Weights & Biases are the most widely known experiment tracking tools, but the ecosystem is vast.  
Depending on your team size, budget, infrastructure, and workflow (classic ML, deep learning, LLMs, RL),  
a different tool may be the best fit.

---

### Why does experiment tracking matter?

```
Without tracking                     With tracking
─────────────────────────────         ─────────────────────────────
 "I think lr=0.001 was best"          Query: runs where val_acc > 0.95
 Lost hyperparameters                 Every run reproducible
 "Which commit trained this?"         Code + data + model linked
 Manual comparison in Excel           Interactive dashboards
 No artifact versioning               Model registry with lineage
```

### Taxonomy of tools covered in this notebook

| Category | Tools |
|---|---|
| Standalone trackers | Neptune.ai, Comet ML, Aim |
| Visualization-first | TensorBoard |
| Configuration-first | Sacred + Omniboard |
| Full MLOps platforms | ClearML, DagHub |
| Cloud-native | Vertex AI Experiments, SageMaker Experiments |
| LLM-specific | LangSmith, Comet LLM, W&B Prompts, Phoenix/Arize, Helicone |
| HPO integration | Optuna, W&B Sweeps, Comet Optimizer, Ax |


## 1. Neptune.ai

Neptune is a **metadata store for ML experiments**. It is designed to handle thousands of runs efficiently  
and is particularly popular among research teams because of its generous free tier.

### Key features
- **Run tracking**: log metrics, parameters, files, images, HTML, interactive plots
- **Metadata management**: every run has a structured namespace (e.g. `run['train/loss']`)
- **Query API** (`neptune.query`): filter, sort, and compare runs programmatically
- **Artifact versioning**: datasets, models, and arbitrary files tracked with content hashes
- **Free tier**: unlimited runs for individuals and researchers

### Architecture

```
Your training script
       │
       │  neptune.init_run()
       ▼
Neptune Client SDK  ──────►  Neptune Server (cloud)
       │                          │
       │  run['metric'].log()     │  stores in columnar DB
       │  run['model'].upload()   │  (TimeSeries + metadata)
       │                          ▼
       │                   Neptune Web UI
       │                   neptune.query API
```

### Run namespace structure

```
run/
  parameters/
    lr          = 0.001
    batch_size  = 32
  train/
    loss        = [time series]
    accuracy    = [time series]
  val/
    loss        = [time series]
  model/
    best.pth    = [File artifact]
  sys/
    creation_time, tags, ...
```


In [1]:
# Neptune.ai: complete run tracking example
# pip install neptune

import neptune
import numpy as np

# ------------------------------------------------------------------
# 1. Initialize a run
# ------------------------------------------------------------------
run = neptune.init_run(
    project="my-workspace/my-project",
    api_token="YOUR_API_TOKEN",          # or set NEPTUNE_API_TOKEN env var
    tags=["baseline", "resnet50"],
    name="resnet50-lr0.001",
)

# ------------------------------------------------------------------
# 2. Log hyperparameters (single values)
# ------------------------------------------------------------------
params = {
    "lr": 0.001,
    "batch_size": 32,
    "optimizer": "adam",
    "epochs": 10,
    "architecture": "resnet50",
}
run["parameters"] = params

# ------------------------------------------------------------------
# 3. Simulate training and log time-series metrics
# ------------------------------------------------------------------
for epoch in range(10):
    # Simulated metrics
    train_loss = 1.0 / (epoch + 1) + np.random.uniform(0, 0.05)
    val_loss   = 1.1 / (epoch + 1) + np.random.uniform(0, 0.07)
    val_acc    = 1 - val_loss / 2

    run["train/loss"].append(train_loss)
    run["val/loss"].append(val_loss)
    run["val/accuracy"].append(val_acc)

# ------------------------------------------------------------------
# 4. Upload model artifact
# ------------------------------------------------------------------
# run["model/best"].upload("best_model.pth")      # upload a file
# run["data/train_csv"].upload("train.csv")        # upload dataset

# ------------------------------------------------------------------
# 5. Log images and HTML
# ------------------------------------------------------------------
# from neptune.types import File
# run["diagnostics/confusion_matrix"].upload(File.as_image(fig))

# ------------------------------------------------------------------
# 6. Stop (flush + close)
# ------------------------------------------------------------------
run.stop()
print("Run finished. View at https://app.neptune.ai/my-workspace/my-project")

[neptune] [warning] NeptuneWarning: By default, these monitoring options are disabled in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', 'capture_hardware_metrics'. You can set them to 'True' when initializing the run and the monitoring will continue until you call run.stop() or the kernel stops. NOTE: To track the source files, pass their paths to the 'source_code' argument. For help, see: https://docs-legacy.neptune.ai/logging/source_code/


NeptuneInvalidApiTokenException: 
[95m
----NeptuneInvalidApiTokenException------------------------------------------------
[0m
The provided API token is invalid.
Make sure you copied and provided your API token correctly.

You can get it or check if it is correct here:
    - https://app.neptune.ai/get_my_api_token

There are two options to add it:
    - specify it in your code
    - set it as an environment variable in your operating system.

[94mCODE[0m
Pass the token to the [1minit_run()[0m function via the [1mapi_token[0m argument:
    [96mneptune.init_run(project='WORKSPACE_NAME/PROJECT_NAME', api_token='YOUR_API_TOKEN')[0m

[94mENVIRONMENT VARIABLE[0m [92m(Recommended option)[0m
or export or set an environment variable depending on your operating system:

    [92mLinux/Unix[0m
    In your terminal run:
        [95mexport NEPTUNE_API_TOKEN="YOUR_API_TOKEN"[0m

    [92mWindows[0m
    In your CMD run:
        [95mset NEPTUNE_API_TOKEN="YOUR_API_TOKEN"[0m

and sk

In [2]:
# Neptune Query API: compare thousands of runs programmatically
import neptune

project = neptune.init_project(
    project="my-workspace/my-project",
    api_token="YOUR_API_TOKEN",
    mode="read-only",
)

# Fetch runs table as a pandas DataFrame
runs_table = project.fetch_runs_table(
    columns=["parameters/lr", "parameters/batch_size", "val/accuracy"],
).to_pandas()

print(runs_table.head())

# Filter: runs where val/accuracy > 0.90
best_runs = runs_table[runs_table["val/accuracy"] > 0.90]
print(f"Runs with val_acc > 0.90: {len(best_runs)}")

# Artifact versioning: Neptune model registry
# model = neptune.init_model(key="CLS", project="my-workspace/my-project")
# model_version = neptune.init_model_version(model="my-workspace/my-project/CLS", version="1.0.0")
# model_version["model"].upload("best_model.pth")

project.stop()

NeptuneInvalidApiTokenException: 
[95m
----NeptuneInvalidApiTokenException------------------------------------------------
[0m
The provided API token is invalid.
Make sure you copied and provided your API token correctly.

You can get it or check if it is correct here:
    - https://app.neptune.ai/get_my_api_token

There are two options to add it:
    - specify it in your code
    - set it as an environment variable in your operating system.

[94mCODE[0m
Pass the token to the [1minit_run()[0m function via the [1mapi_token[0m argument:
    [96mneptune.init_run(project='WORKSPACE_NAME/PROJECT_NAME', api_token='YOUR_API_TOKEN')[0m

[94mENVIRONMENT VARIABLE[0m [92m(Recommended option)[0m
or export or set an environment variable depending on your operating system:

    [92mLinux/Unix[0m
    In your terminal run:
        [95mexport NEPTUNE_API_TOKEN="YOUR_API_TOKEN"[0m

    [92mWindows[0m
    In your CMD run:
        [95mset NEPTUNE_API_TOKEN="YOUR_API_TOKEN"[0m

and sk

## 2. TensorBoard

TensorBoard is Google's visualization toolkit, originally for TensorFlow but now widely used with PyTorch  
via `torch.utils.tensorboard`. It is **local-first** and ships with TF/PyTorch.

### What TensorBoard visualizes

| Plugin | What it shows |
|---|---|
| **Scalars** | Loss, accuracy, LR curves over steps |
| **Histograms** | Weight/gradient distributions over time |
| **Images** | Sample predictions, augmented images |
| **Graphs** | Computation graph (model architecture) |
| **Projector** | High-dim embedding visualization (t-SNE / UMAP) |
| **Profiler** | GPU/CPU utilization, memory, op-level timing |
| **HParams** | Hyperparameter comparison dashboard |
| **Text** | Logged text (e.g., generated captions) |
| **PR Curves** | Precision-Recall curves |

### File layout

```
runs/
  experiment_A/
    events.out.tfevents.1700000000.hostname
  experiment_B/
    events.out.tfevents.1700001000.hostname

$ tensorboard --logdir runs/   # serves at localhost:6006
```

### Integration points

- **TensorFlow 2**: `tf.summary.*` API
- **PyTorch**: `torch.utils.tensorboard.SummaryWriter`
- **PyTorch Lightning**: built-in `TensorBoardLogger`
- **Keras**: `TensorBoard` callback
- **Custom plugins**: build HTML widgets served inside TensorBoard


In [3]:
# TensorBoard with PyTorch: full logging example
# pip install tensorboard torch torchvision

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.tensorboard import SummaryWriter

# ------------------------------------------------------------------
# 1. Define a simple model
# ------------------------------------------------------------------
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)

model     = SimpleNet()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------------
# 2. Create writer: all logs go to runs/mnist_experiment
# ------------------------------------------------------------------
writer = SummaryWriter(log_dir="runs/mnist_experiment")

# ------------------------------------------------------------------
# 3. Log model graph (just needs a dummy input)
# ------------------------------------------------------------------
dummy_input = torch.randn(1, 784)
writer.add_graph(model, dummy_input)

# ------------------------------------------------------------------
# 4. Training loop
# ------------------------------------------------------------------
for step in range(200):
    # Fake batch
    X = torch.randn(32, 784)
    y = torch.randint(0, 10, (32,))

    optimizer.zero_grad()
    logits = model(X)
    loss   = criterion(logits, y)
    loss.backward()
    optimizer.step()

    #, scalars
    writer.add_scalar("Loss/train", loss.item(), step)

    #, weight histograms every 20 steps
    if step % 20 == 0:
        for name, param in model.named_parameters():
            writer.add_histogram(f"weights/{name}", param.data, step)
            if param.grad is not None:
                writer.add_histogram(f"grads/{name}", param.grad, step)

# ------------------------------------------------------------------
# 5. Embedding projector (visualize 2D representation)
# ------------------------------------------------------------------
embeddings = torch.randn(100, 128)   # 100 samples, 128-dim
labels     = [str(i % 10) for i in range(100)]
writer.add_embedding(embeddings, metadata=labels, tag="sample_embeddings")

# ------------------------------------------------------------------
# 6. Log a sample image grid
# ------------------------------------------------------------------
import torchvision
fake_images = torch.randn(16, 1, 28, 28)          # 16 grayscale 28x28 images
img_grid    = torchvision.utils.make_grid(fake_images)
writer.add_image("sample_images", img_grid, 0)

writer.close()
print("Logs written. Run: tensorboard --logdir runs/")

Logs written. Run: tensorboard --logdir runs/


## 3. Comet ML

Comet ML is a cloud experiment tracking platform with emphasis on **project management**, **code tracking**,  
and increasingly on **LLM observability**.

### Standout features
- **Automatic code capture**: every run snapshots the git diff and source files
- **Asset management**: upload datasets, images, confusion matrices, audio
- **Real-time monitoring**: live curves during training
- **Comet LLM**: purpose-built prompt/response tracking for LLM applications
- **Comet Optimizer**: Bayesian HPO integrated with the experiment tracker
- **Panels**: custom JavaScript/Python visualizations embedded in Comet UI

### Comet vs MLflow vs Neptune

```
                 MLflow        Neptune       Comet ML
Self-hostable    Yes           Partial       No (SaaS)
Free tier        Yes           Yes           Yes (limited)
Code tracking    No            Manual        Automatic
LLM support      Via plugins   No            Comet LLM
HPO built-in     No            No            Comet Optimizer
Real-time        No            Yes           Yes
```


In [4]:
# Comet ML: experiment tracking + LLM prompt tracking
# pip install comet_ml comet-llm

# -----------------------------------------------------------
# Part A: Standard experiment tracking
# -----------------------------------------------------------
from comet_ml import Experiment

experiment = Experiment(
    api_key="YOUR_COMET_API_KEY",
    project_name="image-classification",
    workspace="my-workspace",
)

# Log hyperparameters
hyperparams = {
    "learning_rate": 1e-3,
    "batch_size": 64,
    "architecture": "efficientnet_b0",
    "optimizer": "AdamW",
    "weight_decay": 1e-4,
}
experiment.log_parameters(hyperparams)

# Tag the experiment
experiment.add_tags(["baseline", "efficientnet"])

# Log metrics per step
import numpy as np
for epoch in range(5):
    loss = 1.0 / (epoch + 1) + np.random.uniform(0, 0.05)
    acc  = 1 - loss * 0.5
    experiment.log_metric("train_loss", loss, step=epoch)
    experiment.log_metric("val_accuracy", acc, step=epoch)

# Log a confusion matrix (as HTML/image)
# experiment.log_confusion_matrix(y_true, y_pred, labels=class_names)

# Upload model artifact
# experiment.log_model("efficientnet", "./best_model.pth")

experiment.end()

# -----------------------------------------------------------
# Part B: Comet LLM track prompts and LLM responses
# -----------------------------------------------------------
import comet_llm

comet_llm.init(project="llm-experiments", api_key="YOUR_COMET_API_KEY")

comet_llm.log_prompt(
    prompt="Summarize the following article in 3 bullet points: {article}",
    output="• Point 1\n• Point 2\n• Point 3",
    prompt_template="Summarize the following article in 3 bullet points: {article}",
    prompt_template_variables={"article": "Some long article text..."},
    metadata={
        "model": "gpt-4o",
        "temperature": 0.3,
        "latency_ms": 1240,
    },
    tags=["summarization", "v2"],
)
print("Comet LLM prompt logged successfully")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, torch.


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET ERROR: The given API key 'YOUR_COMET_API_KEY' is invalid on 'www.comet.com', please check it against the dashboard. Your experiment will not be logged 
For more details, please refer to: https://www.comet.com/docs/v2/api-and-sdk/python-sdk/warnings-errors/


ModuleNotFoundError: No module named 'comet_ml.config_class'

## 4. Sacred + Omniboard

Sacred is a Python library focused on **configuration management** for experiments not just logging.
It makes every configuration parameter a first-class citizen.

### Philosophy
Sacred treats experiments as **reproducible scientific procedures**:
- Every run has a unique ID, git hash, config snapshot, and captured stdout
- `@ex.config` defines the experiment's configuration space
- `@ex.automain` replaces `if __name__ == '__main__':`
- Observers (MongoDB, file system, SQL) record everything automatically

### Omniboard
Omniboard is a React-based web UI that reads Sacred's MongoDB backend and provides:
- Run table with filtering and sorting
- Metric comparison charts
- Config diffing across runs
- Source code viewer

```
experiment.py
  │  @ex.automain
  │  def run(lr, epochs, _run):
  │      _run.log_scalar("loss", loss)
  │
  └──►  Sacred core
             │
             ▼
        MongoObserver  ──►  MongoDB  ──►  Omniboard UI
        FileStorageObserver ──►  JSON files
```


In [5]:
# Sacred + Omniboard experiment
# pip install sacred pymongo
# Omniboard: npm install -g omniboard && omniboard -m localhost:27017:sacred

from sacred import Experiment
from sacred.observers import MongoObserver, FileStorageObserver
import numpy as np

ex = Experiment("mnist_training", interactive=True)

# ---- Add observers ----
ex.observers.append(
    MongoObserver(url="mongodb://localhost:27017", db_name="sacred")
)
# Also keep a local file backup
ex.observers.append(FileStorageObserver("sacred_runs"))

# ---- Define configuration ----
@ex.config
def cfg():
    lr         = 0.001
    batch_size = 32
    epochs     = 10
    optimizer  = "adam"
    hidden_dim = 256

# ---- Define named configurations (presets) ----
@ex.named_config
def high_lr():
    lr = 0.01

@ex.named_config
def small_model():
    hidden_dim = 64

# ---- Ingredient (reusable config component) ----
@ex.capture
def build_optimizer(model_params, lr, optimizer):
    """Captured function: lr and optimizer come from Sacred config automatically."""
    if optimizer == "adam":
        return {"type": "Adam", "lr": lr}
    return {"type": "SGD", "lr": lr}

# ---- Main training function ----
@ex.automain
def run(lr, epochs, batch_size, hidden_dim, _run):
    """_run is Sacred's run handle for logging."""
    opt_config = build_optimizer(model_params=None)   # lr/optimizer injected
    print(f"Training with lr={lr}, hidden_dim={hidden_dim}, opt={opt_config}")

    best_val = 0
    for epoch in range(epochs):
        train_loss = 1.0 / (epoch + 1) + np.random.uniform(0, 0.05)
        val_acc    = 1 - train_loss * 0.4

        # Sacred scalar logging stored in MongoDB
        _run.log_scalar("train_loss", train_loss, epoch)
        _run.log_scalar("val_accuracy", val_acc, epoch)

        if val_acc > best_val:
            best_val = val_acc

    # Store a final summary metric
    _run.result = best_val
    return best_val

# Usage from command line:
# python experiment.py                    # default config
# python experiment.py with lr=0.01       # override param
# python experiment.py with high_lr       # named config
# python experiment.py print_config       # inspect config without running
print("Sacred experiment defined. Run from command line or call ex.run()")

/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/sacred/dependencies.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


ModuleNotFoundError: No module named 'pymongo'

## 5. Aim: Open-Source, Local-First Experiment Tracking

Aim is a **fully open-source**, self-hosted experiment tracker that stores data locally in a `.aim` repository.  
Its standout feature is **AimQL** a Python-native query language for exploring runs.

### Why Aim?
- No cloud dependency: all data in your `.aim` folder
- Handles thousands of runs with a performant binary format
- Supports multiple experiment types: supervised, RL, CV, NLP
- Sequence comparison: overlay any metric from any run
- AimQL queries feel like native Python

### AimQL Query Examples

```python
# All runs where final val loss < 0.1
runs.filter(lambda r: r['val/loss'] < 0.1)

# Runs tagged 'production' and using Adam
runs.filter(lambda r: 'production' in r.tags and r['hparams', 'optimizer'] == 'Adam')

# Runs from the last 7 days
import datetime
runs.filter(lambda r: r.created_at > datetime.datetime.now() - datetime.timedelta(days=7))

# Sequences: compare val/loss across all runs
metrics.filter(lambda m: m.name == 'val/loss' and m.run['hparams', 'lr'] == 0.001)
```

### Supported experiment types

```
aim.Run       classic train/val loop
aim.Image     log image sequences (CV)
aim.Audio     log audio samples (speech/music ML)
aim.Text      log text sequences (NLP generation)
aim.Figure    log plotly/matplotlib figures
aim.Distribution log weight distributions (RL)
```


In [6]:
# Aim: local experiment tracking with AimQL
# pip install aim
# aim init        (creates .aim repo in current dir)
# aim up          (starts web UI at localhost:43800)

from aim import Run, Image, Text
import numpy as np

# ------------------------------------------------------------------
# 1. Create a run (logs to .aim in current directory)
# ------------------------------------------------------------------
run = Run(
    experiment="cifar10-classification",
    repo="./my-aim-repo",   # optional: shared path for the team
)

# Set hyperparameters (stored as metadata)
run["hparams"] = {
    "lr": 0.001,
    "batch_size": 128,
    "model": "resnet18",
    "optimizer": "Adam",
}
run.add_tag("baseline")
run.add_tag("cifar10")

# ------------------------------------------------------------------
# 2. Training loop log metrics
# ------------------------------------------------------------------
for epoch in range(20):
    train_loss = 2.0 * np.exp(-0.2 * epoch) + np.random.uniform(0, 0.03)
    val_acc    = 1 - train_loss * 0.3

    run.track(train_loss, name="train/loss", epoch=epoch, context={"split": "train"})
    run.track(val_acc,    name="val/accuracy", epoch=epoch, context={"split": "val"})

# ------------------------------------------------------------------
# 3. Log images (e.g., prediction samples)
# ------------------------------------------------------------------
# img = Image(pil_image, caption="prediction at epoch 10")
# run.track(img, name="predictions", step=10)

# ------------------------------------------------------------------
# 4. Log generated text (NLP)
# ------------------------------------------------------------------
# run.track(Text("Generated: The model predicted class 5"), name="generated_text", step=10)

run.close()
print("Run saved to .aim repo")

# ------------------------------------------------------------------
# 5. Query runs with AimQL (programmatic access)
# ------------------------------------------------------------------
from aim import Repo

repo = Repo("./my-aim-repo")

# Filter: runs where val/accuracy > 0.8
query = "run.hparams.lr < 0.01"
for run in repo.query_runs(query).iter_runs():
    print(f"Run {run.hash}: lr={run['hparams', 'lr']}")

# Sequence-level query (filter specific metric sequences)
for metric in repo.query_metrics("metric.name == 'val/accuracy'").iter():
    print(f"  val/accuracy from run {metric.run.hash}: {list(metric.values.sparse_numpy())}")

Run saved to .aim repo


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 524.78it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 253.92it/s]

## 6. ClearML: Full MLOps Platform

ClearML is not just an experiment tracker it is a **complete open-source MLOps platform** covering:

```
┌────────────────────────────────────────────────────────┐
│                   ClearML Platform                     │
├──────────────┬──────────────┬─────────────┬────────────┤
│  Experiment  │     Data     │Orchestration│  Serving   │
│  Tracking    │  Versioning  │  (Queues)   │  (Deploy)  │
│  (Tasks)     │  (Dataset)   │             │            │
├──────────────┴──────────────┴─────────────┴────────────┤
│       Hyperparameter Optimization (HPO)                │
└────────────────────────────────────────────────────────┘
```

### Key concepts
- **Task**: the fundamental unit wraps any Python script automatically
- **Dataset**: versioned dataset with lineage, stored in any backend (S3, GCS, local)
- **Agent**: a worker that pulls tasks from a queue and executes them
- **HPO**: built-in Optuna/Random/Grid search controller that spawns child tasks
- **Pipeline**: DAG of tasks with dependencies

### Auto-logging
ClearML patches popular libraries automatically:
- `matplotlib`, `seaborn` → auto-captured plots
- `sklearn` → model metrics
- `torch` / `tf` → gradient norms, learning rates
- `argparse` → parameters captured from CLI


In [7]:
# ClearML: task management, data versioning, HPO
# pip install clearml
# clearml-init    (configure credentials once)

# -----------------------------------------------------------
# Part A: Basic Task Logging
# -----------------------------------------------------------
from clearml import Task, Logger
import numpy as np

task = Task.init(
    project_name="Image Classification",
    task_name="resnet50-experiment",
    tags=["baseline", "resnet50"],
    auto_connect_frameworks=True,   # auto-log pytorch/tf/sklearn
)

# Connect hyperparameters (also auto-captured from argparse)
params = task.connect({
    "lr": 0.001,
    "batch_size": 32,
    "epochs": 10,
    "architecture": "resnet50",
})

logger = task.get_logger()

# Log scalars
for epoch in range(params["epochs"]):
    loss = 1.0 / (epoch + 1) + np.random.uniform(0, 0.05)
    acc  = 1 - loss * 0.4
    logger.report_scalar("Loss", "train", value=loss, iteration=epoch)
    logger.report_scalar("Accuracy", "val",  value=acc,  iteration=epoch)

# Log a confusion matrix
# logger.report_confusion_matrix("Confusion Matrix", "val", matrix=cm, iteration=10)

task.mark_completed()

# -----------------------------------------------------------
# Part B: ClearML Dataset versioning
# -----------------------------------------------------------
from clearml import Dataset

# Create a versioned dataset
# dataset = Dataset.create(
#     dataset_name="CIFAR-10",
#     dataset_project="Datasets",
# )
# dataset.add_files(path="./data/cifar10/")
# dataset.upload()       # upload to configured storage (S3/local)
# dataset.finalize()     # lock version

# Retrieve a dataset in another script
# ds = Dataset.get(dataset_name="CIFAR-10", dataset_project="Datasets")
# local_path = ds.get_local_copy()

# -----------------------------------------------------------
# Part C: Hyperparameter Optimization
# -----------------------------------------------------------
from clearml.automation import HyperParameterOptimizer, UniformParameterRange, DiscreteParameterRange
from clearml.automation.optuna import OptunaObjective

optimizer = HyperParameterOptimizer(
    base_task_id="BASE_TASK_ID",   # ID of the template task to clone
    hyper_parameters=[
        UniformParameterRange("lr",         min_value=1e-5, max_value=1e-2),
        DiscreteParameterRange("batch_size", values=[16, 32, 64, 128]),
    ],
    objective_metric_title="Accuracy",
    objective_metric_series="val",
    objective_metric_sign="max",
    max_number_of_concurrent_tasks=4,
    optimizer_class=OptunaObjective,
    total_max_jobs=50,
    min_iteration_per_job=5,
    max_iteration_per_job=100,
)

# optimizer.start()   # spins up worker tasks on ClearML Agents
# optimizer.wait()    # block until HPO is complete
# best = optimizer.get_top_experiments(top_k=3)
print("ClearML Task, Dataset, and HPO example defined")

MissingConfigError: It seems ClearML is not configured on this machine!
To get started with ClearML, setup your own 'clearml-server' or create a free account at https://app.clear.ml
Setup instructions can be found here: https://clear.ml/docs

## 7. Cloud-Native: Vertex AI Experiments & SageMaker Experiments

If your infrastructure is already on GCP or AWS, native experiments tools offer tight integration  
with managed training jobs, pipelines, and model registries.

---

### 7.1 Vertex AI Experiments (GCP)

- Integrated with **Vertex AI Training** (custom jobs, AutoML)
- Uses **Cloud Storage** for artifact storage automatically
- Built-in **TensorBoard** integration (managed TensorBoard instance)
- Comparison UI in Google Cloud Console
- Experiments link to **Vertex ML Metadata** for full lineage

```
Vertex Training Job
      │
      │  aiplatform.start_run()
      ▼
Vertex AI Experiments  ──►  Cloud Storage (artifacts)
      │                      Managed TensorBoard
      ▼                      Vertex ML Metadata
Cloud Console UI
```

---

### 7.2 SageMaker Experiments (AWS)

- **Automatic tracking** from SageMaker training jobs (no SDK calls needed)
- Groups runs into **Experiments > Trials > Trial Components**
- Integrates with **SageMaker Studio** for visual comparison
- Links to S3 artifacts, model registry, and SageMaker Pipelines

```
SageMaker Experiment
  └── Trial (one hyperparameter config)
        └── Trial Component (training job / processing job)
              ├── Parameters (hyperparams)
              ├── Metrics    (loss, accuracy)
              └── Artifacts  (model.tar.gz in S3)
```


In [8]:
# Vertex AI Experiments example
# pip install google-cloud-aiplatform

from google.cloud import aiplatform

aiplatform.init(
    project="my-gcp-project",
    location="us-central1",
    experiment="cifar10-classification",
    experiment_tensorboard="projects/my-gcp-project/locations/us-central1/tensorboards/TB_RESOURCE_ID",
)

# Start a run
with aiplatform.start_run("run-001"):
    # Log parameters
    aiplatform.log_params({"lr": 0.001, "batch_size": 32, "model": "resnet50"})

    # Log metrics
    for step in range(10):
        aiplatform.log_metrics({"train_loss": 1.0 / (step + 1), "val_accuracy": 0.7 + step * 0.02})

    # Log time series (for TensorBoard)
    aiplatform.log_time_series_metrics({"lr": 0.001}, step=0)

print("Vertex AI Experiment run logged")

# SageMaker Experiments example
# pip install sagemaker

import boto3
from sagemaker.experiments.run import Run
from sagemaker.session import Session

session = Session(boto_session=boto3.Session(region_name="us-east-1"))

with Run(
    experiment_name="cifar10-experiment",
    run_name="resnet50-baseline",
    sagemaker_session=session,
) as run:
    run.log_parameter("lr", 0.001)
    run.log_parameter("batch_size", 32)

    for step in range(10):
        run.log_metric(name="train:loss",    value=1.0 / (step + 1), step=step)
        run.log_metric(name="val:accuracy",  value=0.7 + step * 0.02, step=step)

    # Artifacts auto-linked when run inside a SageMaker Training Job
    # run.log_artifact(name="best_model", value="s3://my-bucket/models/best.tar.gz")

print("SageMaker Experiment run logged")

ImportError: cannot import name 'aiplatform' from 'google.cloud' (unknown location)

## 8. DagHub: GitHub for ML

DagHub combines **Git + DVC + MLflow** in a single hosted platform, giving you a GitHub-like  
experience specifically designed for ML projects.

### What DagHub provides

```
DagHub Repository
├── Git history          (code, notebooks)
├── DVC remote storage   (large files: data, models)
├── MLflow server        (experiments, runs, artifacts)
└── DagHub UI
    ├── Experiment table (from MLflow)
    ├── Data diff        (DVC diff between commits)
    └── Notebook diff    (visual .ipynb diff)
```

### Workflow

```bash
# 1. Clone a DagHub repo (has both Git and DVC remotes configured)
git clone https://dagshub.com/username/project.git

# 2. Pull large files via DVC
dvc pull

# 3. Run experiments logs to DagHub's hosted MLflow server
export MLFLOW_TRACKING_URI=https://dagshub.com/username/project.mlflow
python train.py

# 4. Push code + data
git push
dvc push
```

### Why DagHub over plain MLflow?
- No infrastructure to manage (hosted MLflow)
- Code + data + experiments linked via the same Git commit
- Public repos are free (great for open-source ML)
- Team collaboration via pull requests on data and models


In [9]:
# DagHub: using MLflow against DagHub's hosted tracking server
# pip install dagshub mlflow dvc

import dagshub
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

# Authenticate and point MLflow to DagHub
dagshub.init(
    repo_owner="my-username",
    repo_name="my-ml-project",
    mlflow=True,           # sets MLFLOW_TRACKING_URI automatically
)

# Now use standard MLflow API runs are stored on DagHub
mlflow.set_experiment("random-forest-baseline")

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {"n_estimators": 100, "max_depth": 5, "min_samples_split": 2}

with mlflow.start_run():
    mlflow.log_params(params)

    model = RandomForestClassifier(**params, random_state=42)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    mlflow.log_metric("accuracy", acc)

    # Log model goes to DagHub's artifact store (backed by DVC)
    mlflow.sklearn.log_model(model, "random-forest")

    print(f"Accuracy: {acc:.4f}")
    print("Run stored at https://dagshub.com/my-username/my-ml-project/experiments")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2e6c222f-6487-42b8-aea4-2f8ee89c8acf&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=8ce83870863359f52e943ab7ef3b450e00ff7f09ac9d2874f82b133a1ae54573




JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## 9. Tool Comparison Matrix

| Tool | Open Source | Storage | Self-hostable | LLM Support | HPO | Free Tier | Best For |
|---|---|---|---|---|---|---|---|
| **MLflow** | Yes (Apache 2) | Local/Cloud | Yes | Via plugins | No | Yes (unlimited) | General ML, self-hosted |
| **W&B** | No (SaaS) | Cloud | No (Teams) | W&B Prompts | Sweeps | Yes (1 user) | Teams, DL, LLMs |
| **Neptune.ai** | Client OSS | Cloud | Partial | No | No | Yes (unlimited runs) | Researchers, large scale |
| **TensorBoard** | Yes (Apache 2) | Local | Yes | No | No | Yes | TF/PyTorch visualization |
| **Comet ML** | No (SaaS) | Cloud | No | Comet LLM | Optimizer | Yes (limited) | Code tracking, LLMs |
| **Sacred** | Yes (MIT) | Local/MongoDB | Yes | No | No | Yes | Config-first, academic |
| **Aim** | Yes (Apache 2) | Local | Yes | No | No | Yes | Local-first, AimQL |
| **ClearML** | Yes (Apache 2) | Local/Cloud | Yes | No | Yes (built-in) | Yes | Full MLOps platform |
| **DagHub** | Client OSS | Cloud | No | No | No | Yes (public) | Git+DVC+MLflow |
| **Vertex AI Exp.** | No (GCP) | GCS | No (GCP only) | No | Via Vertex | Pay-per-use | GCP-native |
| **SageMaker Exp.** | No (AWS) | S3 | No (AWS only) | No | No | Pay-per-use | AWS-native |

### Decision guide

```
Need self-hosting?  ──Yes──►  MLflow, ClearML, Aim, Sacred
       │
      No
       │
       ▼
On GCP?  ──Yes──►  Vertex AI Experiments
On AWS?  ──Yes──►  SageMaker Experiments
       │
      No
       │
       ▼
Tracking LLMs?  ──Yes──►  Comet ML / W&B / LangSmith
       │
      No
       │
       ▼
Need HPO?  ──Yes──►  ClearML, W&B Sweeps, Comet Optimizer
       │
      No
       │
       ▼
Research / free / local?  ──Yes──►  Neptune.ai / Aim
Team collaboration?  ──────────►  W&B / Neptune.ai / DagHub
```


## 10. Hyperparameter Tracking: Optuna, W&B Sweeps, Comet Optimizer, Ax

Experiment trackers and HPO frameworks are naturally complementary.  
The key is to log every trial so you can understand the search landscape.

### HPO Framework Overview

| Framework | Algorithm | Tracker Integration | Distributed | Pruning |
|---|---|---|---|---|
| **Optuna** | TPE, CMA-ES, Grid, Random | MLflow, W&B, Neptune | Yes (RDB backend) | Yes (MedianPruner) |
| **W&B Sweeps** | Bayes, Grid, Random | W&B (built-in) | Yes (agents) | No |
| **Comet Optimizer** | Bayes, Grid, Random | Comet (built-in) | Yes | No |
| **Ax (Meta)** | Bayesian (BOTORCH) | MLflow, W&B | Yes | No |
| **Ray Tune** | Any | MLflow, W&B, Aim | Yes (Ray cluster) | Yes |

### How Optuna + MLflow integration works

```
Optuna Study
  │
  ├── Trial 1: lr=0.001, batch=32  ──►  MLflow Run (trial_0)
  ├── Trial 2: lr=0.01,  batch=64  ──►  MLflow Run (trial_1)
  ├── Trial 3: lr=0.005, batch=32  ──►  MLflow Run (trial_2)  ← pruned
  └── ...
       │
       ▼
  Best trial → registered in MLflow Model Registry
```


In [10]:
# Optuna + MLflow integration: HPO with experiment tracking
# pip install optuna mlflow

import optuna
import mlflow
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score

X, y = make_classification(n_samples=500, n_features=20, random_state=42)

mlflow.set_experiment("optuna-hpo-study")

def objective(trial):
    """Each call is one trial Optuna suggests, we log to MLflow."""
    # Suggest hyperparameters
    lr           = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 50, 500)
    max_depth    = trial.suggest_int("max_depth", 2, 8)
    subsample    = trial.suggest_float("subsample", 0.5, 1.0)

    # Every trial = one MLflow run
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params({
            "lr": lr,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "subsample": subsample,
        })

        model  = GradientBoostingClassifier(
            learning_rate=lr, n_estimators=n_estimators,
            max_depth=max_depth, subsample=subsample,
            random_state=42,
        )
        cv_acc = cross_val_score(model, X, y, cv=3, scoring="accuracy").mean()

        mlflow.log_metric("cv_accuracy", cv_acc)
        mlflow.log_metric("trial_number", trial.number)

        # Pruning: report intermediate value to Optuna
        trial.report(cv_acc, step=0)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return cv_acc

# Run HPO study
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)

with mlflow.start_run(run_name="optuna-study"):
    study.optimize(objective, n_trials=20, n_jobs=1)
    best = study.best_trial
    mlflow.log_metric("best_cv_accuracy", best.value)
    mlflow.log_params({f"best_{k}": v for k, v in best.params.items()})

print(f"Best trial: {best.number}")
print(f"Best accuracy: {best.value:.4f}")
print(f"Best params: {best.params}")

[I 2026-06-19 18:26:41,704] A new study created in memory with name: no-name-34be735d-ce5b-45cc-8651-8ca0b557ce61


COMET INFO: No Comet API Key was found, creating an offline experiment. Set up your API Key to get the full Comet experience https://www.comet.com/docs/python-sdk/advanced/#python-configuration


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, torch.


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET INFO: Using '/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/15_MLOps/.cometml-runs' path as offline directory. Pass 'offline_directory' parameter into constructor or set the 'COMET_OFFLINE_DIRECTORY' environment variable to manually choose where to store offline experiment archives.


...

COMET WARNING: Experiment Name is generated at upload time for Offline Experiments unless set explicitly with Experiment.set_name


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, torch.


COMET INFO: Begin archiving the offline data.


COMET INFO: To upload this offline experiment, run:
    comet upload /home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/15_MLOps/.cometml-runs/3ad3ca5783b540988e539319512c64df.zip


Best trial: 17
Best accuracy: 0.9340
Best params: {'lr': 0.016056167646170152, 'n_estimators': 411, 'max_depth': 3, 'subsample': 0.8309508228823705}


In [11]:
# Ax (Meta's Adaptive Experimentation Platform) + MLflow
# pip install ax-platform mlflow

from ax.service.ax_client import AxClient, ObjectiveProperties
import mlflow
import numpy as np

mlflow.set_experiment("ax-botorch-hpo")

ax_client = AxClient(random_seed=42)
ax_client.create_experiment(
    name="neural_net_tuning",
    parameters=[
        {"name": "lr",           "type": "range", "bounds": [1e-5, 1e-1], "log_scale": True},
        {"name": "hidden_size",  "type": "choice", "values": [64, 128, 256, 512]},
        {"name": "dropout",      "type": "range", "bounds": [0.0, 0.5]},
    ],
    objectives={"val_accuracy": ObjectiveProperties(minimize=False)},
)

for i in range(10):
    params, trial_index = ax_client.get_next_trial()

    # Simulate training
    val_acc = (
        0.7
        + 0.1 * np.log10(params["lr"] / 1e-4)
        + 0.05 * np.log2(params["hidden_size"] / 64)
        - 0.1 * params["dropout"]
        + np.random.normal(0, 0.01)
    )
    val_acc = float(np.clip(val_acc, 0, 1))

    # Log each Ax trial to MLflow
    with mlflow.start_run(run_name=f"ax-trial-{trial_index}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("val_accuracy", val_acc)

    ax_client.complete_trial(trial_index=trial_index, raw_data={"val_accuracy": (val_acc, None)})

best_params, values = ax_client.get_best_parameters()
print(f"Best params: {best_params}")
print(f"Best value: {values}")

[INFO 06-19 18:28:52] ax.storage.sqa_store.with_db_settings_base: Ax SQL storage initialized with SQLAlchemy 2.0.51


/tmp/ipykernel_214382/1986786046.py:10: DeprecationWarning: The `AxClient` class is deprecated and will be removed in Ax 1.4.0. Please migrate to the modern Ax API / `Client` class, found under ax/api. For example usage, check out the tutorials at https://ax.dev 
  ax_client = AxClient(random_seed=42)
[WARNING 06-19 18:28:52] ax.service.ax_client: Random seed set to 42. Note that this setting only affects the Sobol quasi-random generator and BoTorch-powered Bayesian optimization models. For the latter models, setting random seed to the same number for two optimizations will make the generated trials similar, but not exactly the same, and over time the trials will diverge more.


[INFO 06-19 18:28:52] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter lr. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.


[INFO 06-19 18:28:52] ax.service.utils.instantiation: Inferred value type of ParameterType.INT for parameter hidden_size. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/ax/service/utils/instantiation.py:268: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "hidden_size". Defaulting to `True`  since the parameter is not of type string.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
[INFO 06-19 18:28:52] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter dropout. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.


...

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, torch.


COMET INFO: Begin archiving the offline data.


COMET INFO: To upload this offline experiment, run:
    comet upload /home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/15_MLOps/.cometml-runs/ecfc09e8eae0447b9d20c25be0760318.zip


[INFO 06-19 18:29:26] ax.service.ax_client: Completed trial 9 with data: {'val_accuracy': 1.0}.


Best params: {'lr': 0.02396777603956177, 'hidden_size': 512, 'dropout': 0.0}
Best value: ({'val_accuracy': np.float64(1.0049118643190533)}, {'val_accuracy': {'val_accuracy': np.float64(2.175618281501318e-05)}})


## 11. Experiment Reproducibility

Tracking metrics is only half the battle. **Reproducing** a run is what matters for production.

### Reproducibility stack

```
Level 1: Seeds
  random.seed(42)
  numpy.random.seed(42)
  torch.manual_seed(42)
  torch.cuda.manual_seed_all(42)
  torch.backends.cudnn.deterministic = True

Level 2: Environment pinning
  pip freeze > requirements.txt
  conda env export > environment.yml
  poetry lock

Level 3: Code snapshot
  git commit hash logged with every run
  MLflow / Neptune auto-captures source code

Level 4: Data versioning
  DVC, Delta Lake, or Neptune artifact hashes

Level 5: Docker
  FROM pytorch/pytorch:2.1.0-cuda11.8-cudnn8-runtime
  COPY requirements.txt .
  RUN pip install -r requirements.txt
  COPY . /workspace
```

### Common pitfalls
- `torch.backends.cudnn.deterministic = True` alone is not enough also set `benchmark = False`
- Non-determinism from multi-threaded data loaders (`num_workers > 0`) use `worker_init_fn`
- `random.shuffle` order depends on Python version use numpy's seeded RNG instead
- Environment variables like `OMP_NUM_THREADS` affect results on some hardware


In [12]:
# Reproducibility: seeding, environment capture, Docker recipe
import os
import sys
import random
import json
import subprocess
import hashlib
import platform
import numpy as np

# ------------------------------------------------------------------
# 1. Comprehensive seeding utility
# ------------------------------------------------------------------
def seed_everything(seed: int = 42) -> None:
    """Seed all random number generators for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        print(f"[seed] PyTorch seeded with {seed}")
    except ImportError:
        pass

    try:
        import tensorflow as tf
        tf.random.set_seed(seed)
        print(f"[seed] TensorFlow seeded with {seed}")
    except ImportError:
        pass

    print(f"[seed] All RNGs seeded with {seed}")

seed_everything(42)

# ------------------------------------------------------------------
# 2. Environment snapshot (log with every experiment run)
# ------------------------------------------------------------------
def capture_environment() -> dict:
    """Capture the full environment for reproducibility logging."""
    env_info = {
        "python_version": sys.version,
        "platform": platform.platform(),
        "platform_machine": platform.machine(),
    }

    # Get installed packages
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            capture_output=True, text=True, timeout=30
        )
        packages = result.stdout.strip().split("\n")
        env_info["packages"] = packages[:10]  # truncate for demo
        env_info["packages_hash"] = hashlib.md5(result.stdout.encode()).hexdigest()
    except Exception as e:
        env_info["packages_error"] = str(e)

    # Get git commit
    try:
        commit = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        env_info["git_commit"] = commit
        diff_stat = subprocess.run(
            ["git", "diff", "--stat"],
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        env_info["git_dirty"] = bool(diff_stat)
    except Exception:
        env_info["git_commit"] = "not-in-git-repo"

    return env_info

env = capture_environment()
print(json.dumps(env, indent=2))

# ------------------------------------------------------------------
# 3. DataLoader worker seeding (PyTorch)
# ------------------------------------------------------------------
def worker_init_fn(worker_id: int) -> None:
    """Seed each DataLoader worker to ensure data augmentation is reproducible."""
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Usage:
# loader = DataLoader(dataset, num_workers=4, worker_init_fn=worker_init_fn,
#                     generator=torch.Generator().manual_seed(42))

print("\nReproducibility utilities ready")

[seed] PyTorch seeded with 42
[seed] All RNGs seeded with 42


{
  "python_version": "3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]",
  "platform": "Linux-6.17.0-35-generic-x86_64-with-glibc2.39",
  "platform_machine": "x86_64",
  "packages": [
    "a2wsgi==1.10.10",
    "about-time==4.2.1",
    "absl-py==2.4.0",
    "accelerate==1.14.0",
    "adagio==0.2.6",
    "ag-ui-protocol==0.1.19",
    "aim==3.29.1",
    "aim-ui==3.29.1",
    "aimrecords==0.0.7",
    "aimrocks==0.5.2"
  ],
  "packages_hash": "34287665d06962de19e1a6747dde4589",
  "git_commit": "",
  "git_dirty": false
}

Reproducibility utilities ready


## 12. LLM Experiment Tracking

Traditional experiment trackers were built for numeric metrics (loss, accuracy).  
LLM workflows need to track **prompts, completions, token counts, latency, and evaluation scores**.

### The LLM observability stack

```
Prompt versioning layer:
  LangSmith   LangChain-native, traces chains + agents
  W&B Prompts W&B integration for prompt/completion tables
  Comet LLM   standalone prompt tracking, offline-compatible
  Helicone    proxy-based logging (no SDK changes needed)
  Phoenix/Arize open-source LLM observability + evaluation

What to track per LLM call:
  ├── prompt_template  (versioned)
  ├── prompt_variables (inputs)
  ├── model            (gpt-4o, claude-3-opus, ...)
  ├── temperature / top_p / max_tokens
  ├── completion       (output)
  ├── token_counts     (prompt_tokens, completion_tokens)
  ├── latency_ms
  ├── cost_usd
  └── eval_scores      (human rating, LLM-as-judge, RAGAS, ...)
```

### Prompt versioning workflow

```
Prompt v1.0 ──►  A/B test  ──►  Eval scores ──►  Promote to prod
    │                                │
    │  Change system prompt          │  Log to LangSmith / Comet
    ▼                                ▼
Prompt v1.1 ──►  A/B test  ──►  Higher score ──►  Retire v1.0
```


In [13]:
# LLM experiment tracking: LangSmith, Phoenix (Arize), and custom tracking
# pip install langsmith arize-phoenix openai

# -----------------------------------------------------------
# Part A: LangSmith tracing (LangChain)
# -----------------------------------------------------------
import os

# LangSmith is enabled by setting env vars
os.environ["LANGCHAIN_TRACING_V2"]  = "true"
os.environ["LANGCHAIN_API_KEY"]     = "YOUR_LANGSMITH_API_KEY"
os.environ["LANGCHAIN_PROJECT"]     = "summarization-experiments"

# Once these are set, any LangChain code auto-traces
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
#
# llm = ChatOpenAI(model="gpt-4o-mini")
# prompt = ChatPromptTemplate.from_template("Summarize: {text}")
# chain = prompt | llm
# result = chain.invoke({"text": "Some long article..."})
# # ↑ This call is fully traced in LangSmith: inputs, outputs, latency, token count

# -----------------------------------------------------------
# Part B: Phoenix (Arize) open-source LLM observability
# -----------------------------------------------------------
import phoenix as px
from phoenix.trace.openai import OpenAIInstrumentor

# Launch the Phoenix UI (serves at localhost:6006)
session = px.launch_app()

# Auto-instrument OpenAI calls
OpenAIInstrumentor().instrument()

# Now all OpenAI calls are traced:
# from openai import OpenAI
# client = OpenAI(api_key="YOUR_KEY")
# resp = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": "What is MLOps?"}]
# )
# # Visible in Phoenix UI: spans, latency, token usage

print(f"Phoenix UI: {session.url}")

# -----------------------------------------------------------
# Part C: Custom lightweight prompt tracker
# -----------------------------------------------------------
import json
import time
import hashlib
from dataclasses import dataclass, asdict
from typing import Optional
from datetime import datetime

@dataclass
class PromptTrace:
    prompt_template:    str
    prompt_variables:   dict
    rendered_prompt:    str
    completion:         str
    model:              str
    temperature:        float
    prompt_tokens:      int
    completion_tokens:  int
    latency_ms:         float
    eval_score:         Optional[float] = None
    tags:               Optional[list]  = None
    timestamp:          str = ""

    def __post_init__(self):
        self.timestamp = datetime.utcnow().isoformat()
        self.template_hash = hashlib.md5(self.prompt_template.encode()).hexdigest()[:8]

class PromptTracker:
    def __init__(self, log_path: str = "prompt_traces.jsonl"):
        self.log_path = log_path
        self.traces: list[PromptTrace] = []

    def log(self, trace: PromptTrace):
        self.traces.append(trace)
        with open(self.log_path, "a") as f:
            f.write(json.dumps(asdict(trace)) + "\n")

    def get_stats(self) -> dict:
        if not self.traces:
            return {}
        lats   = [t.latency_ms for t in self.traces]
        tokens = [t.prompt_tokens + t.completion_tokens for t in self.traces]
        return {
            "n_calls": len(self.traces),
            "avg_latency_ms": sum(lats) / len(lats),
            "avg_tokens": sum(tokens) / len(tokens),
            "total_cost_estimate_usd": sum(tokens) * 0.000002,
        }

# Demo
tracker = PromptTracker()

trace = PromptTrace(
    prompt_template="Summarize the following in {n} bullet points: {text}",
    prompt_variables={"n": 3, "text": "A long article about ML..."},
    rendered_prompt="Summarize the following in 3 bullet points: A long article about ML...",
    completion="• MLOps automates ML lifecycle\n• Tracking enables reproducibility\n• Monitoring detects drift",
    model="gpt-4o-mini",
    temperature=0.3,
    prompt_tokens=42,
    completion_tokens=28,
    latency_ms=1240.5,
    eval_score=0.85,
    tags=["summarization", "v1.2"],
)

tracker.log(trace)
print(json.dumps(tracker.get_stats(), indent=2))

ImportError: The legacy `phoenix.trace.openai` instrumentor module has been removed.
Please use OpenInference to instrument the OpenAI SDK. Additionally, the `phoenix.otel` module can be used to quickly configure OpenTelemetry:

https://arize.com/docs/phoenix/tracing/integrations-tracing/openai

Example usage:

pip install openinference-instrumentation-openai

```python
from phoenix.otel import register
from openinference.instrumentation.openai import OpenAIInstrumentor

tracer_provider = register()
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
```


## 13. Helicone: Proxy-Based LLM Observability

Helicone is unique: it works as an **HTTP proxy** no SDK changes needed, just change the base URL.  
Every API call is logged automatically.

```python
# Before Helicone
client = OpenAI(api_key="sk-...")

# After Helicone (single line change)
client = OpenAI(
    api_key="sk-...",
    base_url="https://oai.helicone.ai/v1",
    default_headers={"Helicone-Auth": "Bearer HELICONE_API_KEY"},
)
```

Helicone features:
- **Zero code change** for basic tracing
- **Caching**: identical prompts return cached responses (cost savings)
- **Rate limiting**: per-user or per-property rate limits
- **Cost tracking**: per-model, per-user, per-session breakdown
- **Self-hostable** via Docker

## 14. Tracking Across Multiple Frameworks

In production, you often need runs from different frameworks to coexist.  
The adapter pattern lets you swap backends without changing training code:

```
Training code
    │
    │  tracker.log_metric("loss", 0.5)
    ▼
TrackerAdapter (interface)
    ├── MLflowAdapter    ──►  MLflow server
    ├── NeptuneAdapter   ──►  Neptune cloud
    ├── WandbAdapter     ──►  W&B cloud
    └── FileAdapter      ──►  local JSON
```


In [14]:
# Tracker-agnostic adapter pattern
from abc import ABC, abstractmethod
from typing import Any, Dict, Optional
import json
import os
from pathlib import Path

class ExperimentTracker(ABC):
    """Abstract tracker interface swap backends without changing training code."""

    @abstractmethod
    def start_run(self, run_name: str, tags: Optional[Dict] = None) -> None: ...

    @abstractmethod
    def log_params(self, params: Dict[str, Any]) -> None: ...

    @abstractmethod
    def log_metric(self, key: str, value: float, step: Optional[int] = None) -> None: ...

    @abstractmethod
    def log_artifact(self, path: str) -> None: ...

    @abstractmethod
    def end_run(self) -> None: ...

# ---- Implementation: MLflow ----
class MLflowTracker(ExperimentTracker):
    def __init__(self, experiment_name: str, tracking_uri: str = "mlruns"):
        import mlflow
        self.mlflow = mlflow
        mlflow.set_tracking_uri(tracking_uri)
        mlflow.set_experiment(experiment_name)

    def start_run(self, run_name, tags=None):
        self.mlflow.start_run(run_name=run_name, tags=tags or {})

    def log_params(self, params):
        self.mlflow.log_params(params)

    def log_metric(self, key, value, step=None):
        self.mlflow.log_metric(key, value, step=step)

    def log_artifact(self, path):
        self.mlflow.log_artifact(path)

    def end_run(self):
        self.mlflow.end_run()

# ---- Implementation: File-based (no dependencies) ----
class FileTracker(ExperimentTracker):
    """Zero-dependency fallback: logs to JSONL file."""

    def __init__(self, log_dir: str = "runs"):
        self.log_dir   = Path(log_dir)
        self.run_data  = {}
        self.run_name  = None
        self.log_dir.mkdir(parents=True, exist_ok=True)

    def start_run(self, run_name, tags=None):
        self.run_name = run_name
        self.run_data = {"run_name": run_name, "tags": tags or {}, "params": {}, "metrics": []}

    def log_params(self, params):
        self.run_data["params"].update(params)

    def log_metric(self, key, value, step=None):
        self.run_data["metrics"].append({"key": key, "value": value, "step": step})

    def log_artifact(self, path):
        self.run_data.setdefault("artifacts", []).append(path)

    def end_run(self):
        out = self.log_dir / f"{self.run_name}.json"
        with open(out, "w") as f:
            json.dump(self.run_data, f, indent=2)
        print(f"Run saved to {out}")

# ---- Usage completely tracker-agnostic ----
def train(tracker: ExperimentTracker, params: Dict[str, Any]) -> float:
    import numpy as np
    tracker.start_run(run_name="adapter-demo", tags={"env": "notebook"})
    tracker.log_params(params)

    best = 0.0
    for epoch in range(params["epochs"]):
        loss = 1.0 / (epoch + 1) + np.random.uniform(0, 0.05)
        acc  = 1 - loss * 0.4
        tracker.log_metric("train_loss", loss, step=epoch)
        tracker.log_metric("val_acc",    acc,  step=epoch)
        best = max(best, acc)

    tracker.end_run()
    return best

# Use file tracker (works everywhere)
tracker = FileTracker(log_dir="/tmp/demo_runs")
best_acc = train(tracker, {"lr": 0.001, "epochs": 5})
print(f"Best val_acc: {best_acc:.4f}")

# Switch to MLflow by changing one line:
# tracker = MLflowTracker(experiment_name="my-experiment")
# best_acc = train(tracker, {"lr": 0.001, "epochs": 5})

Run saved to /tmp/demo_runs/adapter-demo.json
Best val_acc: 0.9169


## 15. ClearML Task: Production-Grade Logging Pattern

A complete production pattern using ClearML's automatic instrumentation + manual logging + remote execution.


In [15]:
# ClearML: production task with remote execution pattern
# pip install clearml torch torchvision

from clearml import Task
import argparse
import numpy as np

# ------------------------------------------------------------------
# ClearML auto-instruments argparse, logging, matplotlib, sklearn,
# PyTorch, TensorFlow everything is captured without extra code
# ------------------------------------------------------------------
task = Task.init(
    project_name="Production/ImageClassification",
    task_name="efficientnet-v2-training",
    output_uri="s3://my-bucket/clearml-artifacts",   # artifact storage
    tags=["efficientnet", "production"],
)

# Auto-capture argparse parameters
parser = argparse.ArgumentParser()
parser.add_argument("--lr",          type=float, default=1e-3)
parser.add_argument("--epochs",      type=int,   default=50)
parser.add_argument("--batch-size",  type=int,   default=64)
parser.add_argument("--model",       type=str,   default="efficientnet_v2_s")
args = parser.parse_args([])

# ------------------------------------------------------------------
# Execute task remotely on a ClearML Agent instead of locally:
# task.execute_remotely(queue_name="gpu-queue", clone=False, exit_process=True)
# The line above sends the task to a worker and exits locally
# ------------------------------------------------------------------

logger = task.get_logger()

# Simulate training
for epoch in range(5):   # short demo
    train_loss = 2.0 / (epoch + 1)
    val_acc    = 1 - train_loss * 0.3
    val_loss   = train_loss * 1.1

    logger.report_scalar("Loss",     "train",    value=train_loss, iteration=epoch)
    logger.report_scalar("Loss",     "val",      value=val_loss,   iteration=epoch)
    logger.report_scalar("Accuracy", "val",      value=val_acc,    iteration=epoch)

    # Log GPU stats (ClearML also auto-captures GPU metrics if clearml-agent is running)
    logger.report_scalar("GPU",  "utilization_%", value=float(np.random.randint(70, 95)), iteration=epoch)
    logger.report_scalar("GPU",  "memory_gb",     value=float(np.random.uniform(6, 8)),   iteration=epoch)

# Upload best model
# task.upload_artifact("best_model", artifact_object="./checkpoints/best.pth")

# Mark task complete
task.close()
print(f"Task ID: {task.id}")
print("View at https://app.clear.ml")

MissingConfigError: It seems ClearML is not configured on this machine!
To get started with ClearML, setup your own 'clearml-server' or create a free account at https://app.clear.ml
Setup instructions can be found here: https://clear.ml/docs

## 16. Putting It All Together: Multi-Tool Workflow

Real teams often use multiple tools in combination. A common production setup:

```
Development phase:
  ├── Aim (local, fast iteration, AimQL queries)
  └── TensorBoard (real-time loss curves in the IDE)

Team collaboration:
  ├── MLflow or Neptune (central tracking server)
  └── DagHub (code + data + experiments linked)

HPO:
  ├── Optuna (algorithm) + MLflow (tracking each trial)
  └── ClearML HPO (if using ClearML for orchestration)

Production / serving:
  ├── MLflow Model Registry (version models)
  └── ClearML Serving or Vertex AI / SageMaker

LLM applications:
  ├── LangSmith (chain tracing)
  ├── Phoenix/Arize (evaluation + drift)
  └── Helicone (cost + caching)
```

### Golden rules for experiment tracking

1. **Track everything from day 1** retrofitting is painful
2. **Log the git commit hash** with every run
3. **Version your data** (DVC, Delta Lake) data changes break reproducibility
4. **Use descriptive run names and tags** you will thank yourself in 3 months
5. **Store failed runs too** they contain valuable signal
6. **Log system metrics** (GPU util, memory) useful for cost optimization
7. **Use a model registry** for anything that touches production
8. **Automate comparison** don't compare runs manually in spreadsheets


In [16]:
# Final comprehensive example: training script with best-practice tracking
# Demonstrates: seeding + env capture + MLflow + Optuna + model registry

import os
import sys
import random
import json
import numpy as np
import mlflow
import mlflow.sklearn
import optuna
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, classification_report
import platform
import subprocess
import hashlib

# ---- 1. Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ---- 2. Setup MLflow ----
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # sqlite backend supports model registry
mlflow.set_experiment("production-gbt-classifier")

# ---- 3. Environment capture ----
env_info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "seed": SEED,
}

# ---- 4. Data ----
X, y = make_classification(n_samples=2000, n_features=25, n_informative=15,
                            n_classes=3, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# Compute data hash for lineage
data_hash = hashlib.md5(X_train.tobytes()).hexdigest()[:12]

# ---- 5. HPO with Optuna ----
def objective(trial):
    params = {
        "n_estimators":    trial.suggest_int("n_estimators",   50, 300),
        "max_depth":       trial.suggest_int("max_depth",       2, 6),
        "learning_rate":   trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample":       trial.suggest_float("subsample",     0.6, 1.0),
        "min_samples_leaf":trial.suggest_int("min_samples_leaf", 1, 10),
    }

    with mlflow.start_run(run_name=f"trial-{trial.number:03d}", nested=True) as run:
        # Log everything
        mlflow.set_tags({"env": json.dumps(env_info), "data_hash": data_hash,
                         "optuna_study": "gbt-hpo"})
        mlflow.log_params(params)
        mlflow.log_param("seed", SEED)

        model  = GradientBoostingClassifier(**params, random_state=SEED)
        cv_acc = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").mean()
        cv_std = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy").std()

        mlflow.log_metric("cv_accuracy_mean", cv_acc)
        mlflow.log_metric("cv_accuracy_std",  cv_std)

    return cv_acc

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))

with mlflow.start_run(run_name="hpo-study") as parent_run:
    mlflow.set_tag("run_type", "hpo_parent")
    study.optimize(objective, n_trials=10, n_jobs=1)

    # ---- 6. Retrain best model on full training set ----
    best_model = GradientBoostingClassifier(**study.best_params, random_state=SEED)
    best_model.fit(X_train, y_train)

    test_acc = accuracy_score(y_test, best_model.predict(X_test))
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_cv_accuracy", study.best_value)
    mlflow.log_metric("test_accuracy",    test_acc)
    mlflow.log_param("n_trials", len(study.trials))

    # ---- 7. Register model ----
    mlflow.sklearn.log_model(
        best_model,
        artifact_path="gbt_model",
        registered_model_name="production-gbt-classifier",
    )

    print(f"Best CV accuracy:  {study.best_value:.4f}")
    print(f"Test accuracy:     {test_acc:.4f}")
    print(f"Best params: {study.best_params}")
    print(f"MLflow run ID: {parent_run.info.run_id}")

[I 2026-06-19 18:29:33,905] A new study created in memory with name: no-name-9931f208-daf9-4de0-b414-e7a7b99c4ba9


COMET INFO: No Comet API Key was found, creating an offline experiment. Set up your API Key to get the full Comet experience https://www.comet.com/docs/python-sdk/advanced/#python-configuration


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, torch.


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.


COMET INFO: Using '/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/15_MLOps/.cometml-runs' path as offline directory. Pass 'offline_directory' parameter into constructor or set the 'COMET_OFFLINE_DIRECTORY' environment variable to manually choose where to store offline experiment archives.


...

COMET WARNING: Experiment Name is generated at upload time for Offline Experiments unless set explicitly with Experiment.set_name


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: tensorboard, torch.


Best CV accuracy:  0.8344
Test accuracy:     0.8275
Best params: {'n_estimators': 144, 'max_depth': 6, 'learning_rate': 0.06504856968981275, 'subsample': 0.8394633936788146, 'min_samples_leaf': 2}
MLflow run ID: de537d2752c04bf1b28ef14f4b671bf4


COMET INFO: Begin archiving the offline data.


COMET INFO: To upload this offline experiment, run:
    comet upload /home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/15_MLOps/.cometml-runs/124a0402914f4786b0f739f4cf20be30.zip


## Additional Learning Resources

### Papers

1. **"Scaling MLOps: The Hidden Technical Debt in Machine Learning Systems"**  
   Sculley et al., NIPS 2015 the foundational paper on ML operational debt.  
   https://papers.nips.cc/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html

2. **"Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology"**  
   Studer et al., 2021 structured process for reproducible ML.  
   https://arxiv.org/abs/2003.05155

3. **"A Survey of Hyperparameter Optimization Methods"**  
   Feurer & Hutter, 2019 comprehensive HPO taxonomy.  
   https://link.springer.com/chapter/10.1007/978-3-030-05318-5_1

4. **"Optuna: A Next-generation Hyperparameter Optimization Framework"**  
   Akiba et al., KDD 2019.  
   https://arxiv.org/abs/1907.10902

### Official Documentation

- **Neptune.ai docs**: https://docs.neptune.ai
- **TensorBoard guide**: https://www.tensorflow.org/tensorboard/get_started
- **Comet ML docs**: https://www.comet.com/docs
- **Aim docs**: https://aimstack.readthedocs.io
- **ClearML docs**: https://clear.ml/docs
- **Sacred docs**: https://sacred.readthedocs.io
- **LangSmith docs**: https://docs.smith.langchain.com
- **Phoenix/Arize**: https://docs.arize.com/phoenix
- **Helicone**: https://docs.helicone.ai

### Courses and Tutorials

- **Made With ML (MLOps)**: https://madewithml.com end-to-end MLOps with tracking
- **Full Stack Deep Learning**: https://fullstackdeeplearning.com lecture on experiment management
- **Weights & Biases Courses**: https://wandb.ai/fully-connected free courses on W&B
- **Neptune.ai Blog**: https://neptune.ai/blog in-depth experiment tracking tutorials
- **Evidently AI Blog**: https://www.evidentlyai.com/blog ML monitoring (complements tracking)

### Books

- **"Designing Machine Learning Systems"** Chip Huyen (O'Reilly 2022) Chapter 6: Feature Engineering + Chapter 9: Continual Learning
- **"Machine Learning Engineering"** Andriy Burkov (2020) Chapter 8: Model Serving and Monitoring
- **"Practical MLOps"** Noah Gift & Alfredo Deza (O'Reilly 2021) Chapter 5: AutoML and KPIs

### Community

- **MLOps Community Slack**: https://mlops.community
- **Papers with Code (MLOps)**: https://paperswithcode.com/task/model-training
- **Awesome MLOps GitHub**: https://github.com/visenger/awesome-mlops
- **Neptune.ai Experiment Tracking Comparison** (updated): https://neptune.ai/blog/best-ml-experiment-tracking-tools


## Summary

This notebook covered the full experiment tracking ecosystem:

| # | Tool / Topic | Key Takeaway |
|---|---|---|
| 1 | Neptune.ai | Metadata store; query API for 1000s of runs; free for researchers |
| 2 | TensorBoard | Local-first visualization; histograms, embeddings, profiler |
| 3 | Comet ML | Auto code tracking; Comet LLM for prompt versioning |
| 4 | Sacred + Omniboard | Config-first; `@ex.automain`; MongoDB backend |
| 5 | Aim | Open-source, local; AimQL for expressive run queries |
| 6 | ClearML | Full MLOps: tasks + data + orchestration + HPO + serving |
| 7 | Vertex AI / SageMaker | Cloud-native; auto-tracks managed training jobs |
| 8 | DagHub | Git + DVC + MLflow in one place; great for open-source ML |
| 9 | Comparison matrix | Choose based on: self-hosting, cloud, LLM, HPO needs |
| 10 | Optuna + Ax + HPO | Log every trial; use nested runs; connect to tracker |
| 11 | Reproducibility | Seeds + env pin + git hash + data version + Docker |
| 12 | LLM tracking | LangSmith, Phoenix, Helicone, Comet LLM |
| 13 | Adapter pattern | Decouple training code from tracker implementation |
| 14 | Production pattern | Combine MLflow registry + Optuna + ClearML agents |

### Next steps

- Set up a **local ClearML server** and migrate from MLflow
- Integrate **Aim** into a current project for offline-first tracking
- Add **LangSmith** to any LangChain application you maintain
- Implement the **adapter pattern** to make your training code tracker-agnostic
- Read the **Sculley et al. (2015)** paper for the full picture of ML technical debt
